# Tutorial: running the repo's code on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/tutorial_colab_oracle.ipynb)

We write and version the code locally (VS Code + GitHub), and use Colab only to **run** it on a GPU.
This notebook shows that workflow end to end. As a test workload, it loads Qwen3-8B and an
Activation Oracle and asks the oracle one question.

**How to run it**

1. Open the notebook in Colab: click the badge above, or *File → Open notebook →
   GitHub*, paste the repo URL, pick your branch and select this notebook.
2. Select a GPU: *Runtime → Change runtime type → GPU* (the free T4 is enough).
3. If you are testing a branch, set `BRANCH` in the setup cell below.
4. *Runtime → Run all*. The first run takes ~5 min to download Qwen3-8B (16 GB).

**Day-to-day workflow**

- Edit code locally, commit and push to your branch. In Colab, rerun the setup cell: it pulls the
  latest version of the branch.
- Edits made in Colab are **not** saved to the repo. Keep important code in `src/procom/`, not here.
- A Colab runtime is wiped when it disconnects (idle timeout, end of session): installed packages,
  downloaded models and any outputs are lost. Rerun the setup cell, and save results you want to
  keep to Google Drive.

## 0. Setup

This cell does the following:
- clone the repo on Colab and check out the selected `BRANCH`.
- install the dependencies on Colab.

Rerun this cell every time you open a Colab session and after every push on GitHub.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # put your branch here to test code that is not merged yet

if IN_COLAB:
    # Clone the repo (or update it), then install the procom package and its dependencies
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # makes procom importable without restarting

In [ ]:
import torch

import procom  # noqa: F401  (checks that the repo's package is installed)

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

## 1. Loading Qwen3-8B and the oracle

The Activation Oracle ([Karvonen et al., 2025](https://arxiv.org/abs/2512.15674)) is an LLM trained to
read another LLM's activations and answer questions about them in natural language. For Qwen3-8B, it is
a **LoRA adapter trained on Qwen3-8B itself**, so the model is loaded only once:
LoRA disabled = original Qwen3-8B, LoRA enabled = oracle.
Reference code: [adamkarvonen/activation_oracles](https://github.com/adamkarvonen/activation_oracles)
(notebook `experiments/activation_oracle_demo.ipynb`).

Qwen3-8B in bf16 takes ~16 GB: it does not fit on a T4 (15 GB), so in that case we load it in 8-bit.

In [ ]:
import logging

import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"
ORACLE_LORA = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
load_in_8bit = gpu_memory_gb < 20
quantization = BitsAndBytesConfig(load_in_8bit=True) if load_in_8bit else None
# In 8-bit, bitsandbytes logs a harmless "inputs will be cast to float16" warning at every
# forward pass: we hide it (and only bitsandbytes warnings)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto", quantization_config=quantization
)
device = model.device
layers = model.model.layers  # the transformer blocks, where activations are read and injected

model = PeftModel.from_pretrained(model, ORACLE_LORA)  # adds the oracle's LoRA
model.eval()
torch.set_grad_enabled(False)

print(f"GPU {gpu_memory_gb:.0f} GB | 8-bit: {load_in_8bit} | {len(layers)} layers")

## 2. Qwen3-8B alone

First check: the target LLM answers a question. Inside `model.disable_adapter()`, the LoRA is turned
off: this is the original Qwen3-8B.

In [ ]:
def chat(messages, max_new_tokens=60):
    """Has Qwen3-8B (without the oracle) answer a conversation."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    with model.disable_adapter():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1] :], skip_special_tokens=True)


print(chat([{"role": "user", "content": "What is the capital of France? Answer in one sentence."}]))

## 3. Reading Qwen3-8B's activations

A PyTorch *forward hook* grabs the output of a layer (the residual stream) during a forward pass.
The oracle was trained on activations taken at 25, 50 and 75 % depth; as in the paper, we use
50 %, i.e. layer 18 of Qwen3-8B. The next cell runs it on an example prompt and prints the tokens
with their indices, one activation vector per token.

In [ ]:
ACT_LAYER = len(layers) // 2  # 50 % depth: layer 18 for Qwen3-8B


def get_activations(messages, layer=ACT_LAYER):
    """Activations of Qwen3-8B (without LoRA) at the output of `layer`, one row per token."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    captured = {}

    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["acts"] = hidden[0].detach()  # (n_tokens, d_model)

    handle = layers[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter():
            model(**inputs)
    finally:
        handle.remove()  # always remove the hook, otherwise it stays active afterwards
    tokens = [tokenizer.decode(t) for t in inputs.input_ids[0]]
    return captured["acts"], tokens

In [ ]:
target = [
    {
        "role": "user",
        "content": "The philosopher who drank hemlock taught a student who founded an academy. "
        "That student's most famous pupil was",
    }
]
acts, tokens = get_activations(target)
print("activations:", tuple(acts.shape))
print(list(enumerate(tokens)))

## 4. Asking the oracle

The oracle's prompt uses the format it was trained on:

```
Layer: 18
 ? ? ? … ?
<question>
```

with one ` ?` token per activation vector. During the oracle's forward pass, a hook at the output of
**layer 1** adds each vector `v` onto its ` ?`, rescaled to the norm of the residual `h` at that
position: `h ← h + ‖h‖ · v / ‖v‖`.

In [ ]:
INJECT_LAYER = 1  # the oracle was trained with injection at the output of layer 1
PLACEHOLDER = " ?"


def ask_oracle(acts, question, max_new_tokens=50):
    """Asks the oracle `question` about the activation vectors `acts` (n_vectors, d_model)."""
    n = len(acts)
    prompt = f"Layer: {ACT_LAYER}\n" + PLACEHOLDER * n + " \n" + question
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)

    # Positions of the n " ?" tokens in the prompt (the first n: the question may contain some too)
    placeholder_id = tokenizer.encode(PLACEHOLDER, add_special_tokens=False)[0]
    positions = (inputs.input_ids[0] == placeholder_id).nonzero().flatten()[:n]
    assert len(positions) == n and positions[-1] - positions[0] == n - 1

    def inject(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.shape[1] == 1:  # generation steps (1 token at a time): nothing to inject
            return
        h = hidden[0, positions]
        v = F.normalize(acts, dim=-1).to(h.dtype)
        hidden[0, positions] = h + h.norm(dim=-1, keepdim=True) * v

    handle = layers[INJECT_LAYER].register_forward_hook(inject)
    try:
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        handle.remove()
    return tokenizer.decode(output[0, inputs.input_ids.shape[1] :], skip_special_tokens=True)

We ask the same question with two inputs: all the activations of the prompt, then only the vector
of the ` pupil` token. On this example, the authors' demo gets, for the ` pupil` token:
*"The assistant is currently thinking about Socrates, Plato, and Aristotle."*

In [ ]:
question = "Can you name which person the model is thinking about?"
pupil = tokens.index(" pupil")

print("All positions:     ", ask_oracle(acts, question))
print("' pupil' token only:", ask_oracle(acts[pupil : pupil + 1], question))